# Architecture in the cloud

A tested service in a container (notebook 02) is still not available to its users. This first lecture notebook of Block 2 introduces **system design and cloud architecture**: how a front end, a back end and a database work together, how Docker Compose runs them on one machine, and which cloud services run them for real users. A last, optional section shows simple ways of **publishing a dashboard**. The second lecture notebook of the block, [05-lecture-monitoring-and-maintenance.ipynb](05-lecture-monitoring-and-maintenance.ipynb), asks how the deployed model stays useful.

The parts of the tariff application and how they talk to each other:

```mermaid
flowchart LR
    U["User<br/>(browser)"] -->|HTTPS| FE["Front end<br/>Streamlit dashboard"]
    FE -->|"HTTP + JSON<br/>POST /predict"| BE["Back end<br/>FastAPI + model"]
    P["Other programs"] -->|"HTTP + JSON"| BE
    BE -->|SQL| DB[("Database<br/>SQLite file")]
```

## System design and cloud architecture

This section is an introduction. Its aim is that you can read and sketch the architecture of a data application and know which cloud services exist for each part, not that you operate them.

### Concept

**System design** is the decision about which parts an application consists of, what each part does and how the parts communicate. Most data applications follow the same three-part pattern:

| Part | What it does | In the tariff example |
|---|---|---|
| **Front end** | What users see and click: a web page or a dashboard | the Streamlit dashboard, where an officer pastes a description |
| **Back end** | The logic behind it, offered as an API: validation, the model, the rules | the FastAPI service that returns the top three headings |
| **Database** | Data that must outlive a single request | a SQLite file with the prediction log, the raw data of every drift check (notebook 05) |

The parts talk over the network: the front end sends HTTP requests with JSON to the back end (Session 2), the back end sends SQL to the database (Session 3). Only the front end and the API are reachable from outside; the database is not.

(The diagram at the top of this page shows the three parts.)

Why separate the parts instead of putting everything into one script?

- **One model, several clients.** The dashboard, a batch job and another team's software can all use the same API, and a new model version is deployed in one place.
- **Independent scaling.** If many requests arrive, more copies of the back end are started; the database stays one.
- **Security.** The database holds the data and is reachable only from the back end, never from the internet.
- **Replaceable parts.** The Streamlit front end can later be replaced by a web page without touching the model.

Not every project needs all three parts. Three common designs:

| Design | Parts | Example |
|---|---|---|
| Interactive model service | front end → back end → database | the tariff heading service in this workspace |
| Analytics dashboard | front end → database, refreshed by a scheduled job | a dashboard of Berlin rental prices, reloaded every night by the loading script |
| Batch scoring | scheduled job → database → reporting tool | churn scores computed every Monday for all customers and read by the CRM team |

**Locally**, front end and back end each run in their own container, and **Docker Compose** starts them together from one file, `compose.yaml`. The containers share a private network in which they find each other by service name: the dashboard calls `http://api:8000`. The database is a **SQLite** file (Session 3) on a **volume**, a storage area that Docker keeps when a container is replaced; only the API container mounts it, so the log survives a restart and cannot be reached from outside. One SQLite file serves one API container well. When several copies of the back end on different machines must write to the same data, the SQLite documentation recommends a client/server database instead, and that is what the cloud offers as a managed service.

**In the cloud**, the same parts run on managed services. A provider such as **Amazon Web Services (AWS)** offers a service for each part:

| Part | Locally (Compose) | AWS service | What the service does |
|---|---|---|---|
| Container images | `docker build` | **Amazon ECR** (Elastic Container Registry) | stores the images that CI builds |
| Front end and back end | containers `dashboard`, `api` | **Amazon ECS with AWS Fargate** | runs containers without managing servers; starts more copies under load |
| Entry point | `ports:` | **Elastic Load Balancing** (Application Load Balancer) | receives HTTPS traffic and distributes it to the containers |
| Database | SQLite file on the volume `prediction-data` | **Amazon RDS** (Relational Database Service) | a managed SQL database server with backups and updates, for several back-end copies |
| Files: model, raw data | `models/` folder | **Amazon S3** | object storage for files of any size |
| Passwords and keys | `environment:` | **AWS Secrets Manager** | stores secrets and passes them to the containers |
| Logs and metrics | `docker logs` | **Amazon CloudWatch** | collects logs, metrics and alarms |

```mermaid
flowchart LR
    GH["GitHub Actions<br/>(cd.yml)"] -->|push image| ECR["Amazon ECR"]
    U["Users"] -->|HTTPS| LB["Load balancer"]
    subgraph ECS["Amazon ECS on Fargate"]
        FE["dashboard<br/>container"]
        BE["api<br/>containers"]
    end
    ECR -.->|pull| ECS
    LB --> FE
    LB --> BE
    FE --> BE
    BE --> RDS[("Amazon RDS<br/>SQL database")]
    BE -.->|model files| S3["Amazon S3"]
    ECS -.->|logs| CW["Amazon CloudWatch"]
```

Microsoft Azure and Google Cloud offer the same building blocks under other names, for example Azure Container Apps or Google Cloud Run for the containers and Azure SQL Database or Google Cloud SQL for the database. The design stays the same; only the names change.

### Why it matters

Data scientists rarely set up cloud infrastructure alone, but they take part in the decisions: where the model runs, where the data live, who can reach them and what it costs. A sketch of the architecture is the common language for these discussions with engineers, IT security and the budget holder. It also makes data protection visible: in the diagram you can point to the one place where confidential requests are stored.

### How it works in Python

The workspace contains the three-part version of the service. `compose.yaml`, shortened:

```yaml
services:
  api:                                    # back end: the image of the Dockerfile (notebook 02)
    build: .
    environment: {PREDICTION_DB: /app/data/predictions.sqlite}
    volumes: ["prediction-data:/app/data"]   # database: a SQLite file that only the API sees
    ports: ["8000:8000"]
  dashboard:                              # front end: sends every description to the API
    build: dashboard
    environment: {API_URL: "http://api:8000"}
    ports: ["8501:8501"]
volumes:
  prediction-data:
```

```bash
cd sessions/16-deployment-and-monitoring/workspace
docker compose up --build        # dashboard: http://localhost:8501, API: http://localhost:8000/docs
docker compose exec api python -c "import sqlite3; print(sqlite3.connect('/app/data/predictions.sqlite').execute('SELECT heading, language, model_version FROM predictions').fetchall())"
docker compose down              # add -v to delete the volume with the log as well
```

The service code itself hardly changes: with the environment variable `PREDICTION_DB` set, the API writes its prediction log to a SQLite table instead of a file of JSON lines (`monitoring.py`), and with `API_URL` set, the dashboard calls the API instead of loading the model (`dashboard/streamlit_app.py`). Settings that differ between laptop and cloud come from the environment, not from the code (*The Twelve-Factor App*).

### In practice

- AWS, Azure and Google Cloud each publish a "well-architected framework" with design principles for reliability, security, cost and performance; the AWS Well-Architected Framework is a common reference in job interviews and design reviews.
- Many companies run their containers on Kubernetes (Amazon EKS, Azure AKS, Google GKE) once they have many services; for a few services, managed container platforms such as ECS on Fargate are simpler.
- Architecture diagrams with the providers' official icons are a standard part of project documentation and of the review before a system goes live.

> **Caution.** Cloud resources cost money while they exist, whether or not anyone uses them: a load balancer and a database are billed by the hour. The free offers for new accounts are limited in time and amount, and their terms change; check them, set a budget alarm, and delete what you created after an experiment.

## Publishing a dashboard (optional)

### Concept

Not every model needs an API. For many projects, the users are people who want to explore results: a **dashboard** is a web page with charts, filters and perhaps a form that calls the model. **Streamlit** turns a Python script into such a web app: every widget (`st.text_area`, `st.selectbox`) is a line of Python, and the script reruns when the user changes an input. `@st.cache_resource` keeps the loaded model in memory between reruns.

Publishing options differ in what they run and what they cost. Free tiers change often; check the current terms before you choose.

| Option | Runs | Fits | Notes |
|---|---|---|---|
| Streamlit Community Cloud | Streamlit apps from a public GitHub repository | team dashboards, course projects | free; the app sleeps when unused; files must be in the repository or downloaded at start |
| Hugging Face Spaces | Gradio, Streamlit, static and Docker apps | model demos | free hardware only for some app types; paid hardware for others |
| Render, Google Cloud Run and similar | any container image | the API container | free quotas with limits; some need a billing account |
| A university or company server | containers with Docker or Podman | internal or personal data | data stay inside the organisation |
| A static report (Quarto, Jupyter Book on GitHub Pages) | HTML without a server | results that do not need live predictions | cheapest and most robust |

A decision guide:

```mermaid
flowchart TD
    Q{"Do users need live<br/>predictions?"} -->|no| ST["Static report:<br/>GitHub Pages"]
    Q -->|yes| P{"Are the users<br/>people or programs?"}
    P -->|programs| API["API container:<br/>Cloud Run, Render, server"]
    P -->|people| D{"Personal or<br/>confidential data?"}
    D -->|yes| IN["Internal server<br/>or company cloud"]
    D -->|no| SC["Streamlit Community Cloud<br/>or Hugging Face Spaces"]
```

### Why it matters

A model that nobody can use has no effect. For a project manager, choosing where and how results are published is a decision about cost, maintenance and data protection: a public free tier is fine for a demo on published BTI decisions, not for pending requests, which contain confidential business information.

### How it works in Python

`workspace/dashboard/streamlit_app.py`, the core:

```python
# requires: uv sync --extra dashboard, then: MODEL_DIR=models uv run streamlit run dashboard/streamlit_app.py
import os
from pathlib import Path

import pandas as pd
import streamlit as st

from tariff_service.model import decision_text, load_model, top_k

@st.cache_resource                      # load once per server process
def get_model():
    return load_model(Path(os.environ.get("MODEL_DIR", "models")))

pipe, meta = get_model()
st.title("Tariff heading suggestion")
st.caption(f"Model {meta['model_version']} · validation accuracy {meta['validation']['accuracy']}")
text = st.text_area("Description of goods", "Masque de protection à usage unique, en non-tissé")
ranked = top_k(pipe, [decision_text(text)], k=3)[0]          # [(heading, score), ...]
st.dataframe(pd.DataFrame([{"heading": h, "score": sc, "text": meta["headings"].get(h, "")}
                           for h, sc in ranked]))
```

To publish it on Streamlit Community Cloud: push the repository to GitHub (public), sign in at [share.streamlit.io](https://share.streamlit.io/) with GitHub, choose the repository and the file `dashboard/streamlit_app.py`, and add the dependencies in a `requirements.txt` next to the app or in `pyproject.toml`. The model files must be reachable: commit a small model (a few MB), or download it from the GitHub release at start-up.

### In practice

- The COVID-19 dashboard of Johns Hopkins University (Dong, Du and Gardner, 2020) showed how a regularly updated public dashboard can become a primary information source.
- Our World in Data publishes its charts and the code behind them openly, and links every chart to its data source.
- Many data science teams publish internal Streamlit or Dash apps for colleagues instead of sending notebooks.

> **Warning.** A public dashboard is public: anyone can use it, and every prediction costs computing time. Do not publish personal data, and do not connect a public app to a paid API without limits.

## Check your understanding

1. Your team's dashboard classifies pending BTI requests that customs officers paste into it. Which publishing option do you choose, and why?
2. Sketch the architecture of your team project: which parts does it need (front end, back end, database, scheduled job), and which AWS service would run each part?
3. Why does `compose.yaml` publish ports for the dashboard and the API, and why is the SQLite file on a volume that only the API container mounts?
4. The API is so popular that three copies of it run behind a load balancer. Why is one SQLite file no longer a good place for the prediction log, and what replaces it?

## Further reading

- Docker Inc. *Docker Compose: How Compose works*. [docs.docker.com/compose/intro/compose-application-model](https://docs.docker.com/compose/intro/compose-application-model/)
- Amazon Web Services. *AWS Well-Architected Framework*. [docs.aws.amazon.com/wellarchitected/latest/framework/welcome.html](https://docs.aws.amazon.com/wellarchitected/latest/framework/welcome.html)
- Amazon Web Services. *What is Amazon Elastic Container Service?* [docs.aws.amazon.com/AmazonECS/latest/developerguide/Welcome.html](https://docs.aws.amazon.com/AmazonECS/latest/developerguide/Welcome.html)
- Streamlit. *Deploy your app on Community Cloud*. [docs.streamlit.io/deploy/streamlit-community-cloud](https://docs.streamlit.io/deploy/streamlit-community-cloud)
- SQLite. *Appropriate uses for SQLite*. [sqlite.org/whentouse.html](https://www.sqlite.org/whentouse.html)